In [1]:
!pip uninstall -y transformers
!pip install -q transformers==4.41.2 sentencepiece accelerate

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 605.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.7 MB/s eta 0:00:00


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModel,pipeline
#from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import transformers

In [4]:
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
def create_combined_text(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

dataset = dataset.map(create_combined_text)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
#Question-1:

"""Load train.csv using the Hugging Face datasets library (do not use pandas). 
Use the .map() function to create a new column called combined_text that concatenates the 
prompt and A columns with a space in between. E.g., prompt_text A_text. What is the 
exact character length (total number of string characters using Python's len() 
function, NOT the number of tokens) of the combined_text string for the row at index 51?
Note: We follow zero-indexing here.
"""

print(dataset[51]["combined_text"])
print(len(dataset[51]["combined_text"]))

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
614


In [7]:
#Question-2:
"""Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration 
properties: what is the exact total vocabulary size (the maximum number of unique subword 
tokens the model knows) hardcoded into this tokenizer? """

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(tokenizer.vocab_size)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


In [8]:
#Question-3:
"""Transformers rely on special tokens to understand sentence boundaries. 
Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID 
assigned to the [SEP] (Separator) token.  """

print(tokenizer.sep_token)
print(tokenizer.sep_token_id)

[SEP]
102


In [9]:
#Question-4:
"""Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train 
dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and 
return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?"""

prompts = list(dataset["prompt"])

encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encoded["input_ids"].shape)

torch.Size([2000, 128])


In [10]:
#Question-5:

"""A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and 
uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. 
What is the exact dimensionality (size) of each individual attention head?  """

print(768//12)

64


In [11]:
#Question-6:

"""Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt 
from row ID 0 using the tokenizer's default settings (do not apply any manual padding or 
truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here."""

model = AutoModel.from_pretrained("bert-base-uncased")

# Prompt from row 0
text = dataset[0]["prompt"]

# Default tokenizer settings (no padding/truncation arguments)
inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

torch.Size([1, 31, 768])


In [12]:
#Question-7:

"""Using the last_hidden_state tensor from the previous question, extract the embedding 
vector representing the [CLS] token (which is always the token at index 0). What is the sum 
of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places). """

# Extract the [CLS] embedding (first token of the first sequence)
cls_embedding = outputs.last_hidden_state[0, 0]

# Sum the first 5 values
result = cls_embedding[:5].sum().item()

print(round(result, 4))

-1.2001


In [13]:
#Question-8:

"""Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact 
string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it
through the model. Extract the attention matrix for the last layer (index -1) and the first 
attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays
to the word fusion (you will need to find the specific token index for fusion in the 
input_ids)? (Round your answer to 4 decimal places).  """


# Load model with attentions
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")

# See the tokens and their indices
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
for i, token in enumerate(tokens):
    print(i, token)

outputs = model(**inputs)

# Find the index of "fusion"
fusion_idx = tokens.index("fusion")

# Last layer, first head
attention = outputs.attentions[-1][0, 0]

print("Fusion index:", fusion_idx)
print("Attention weight:", round(attention[0, fusion_idx].item(), 4))

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]
Fusion index: 4
Attention weight: 0.1025


In [14]:
# #Question-9:

# """Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() 
# method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the 
# cosine similarity between these two vectors specifically using the 
# sentence_transformers.util.cos_sim() function. What is the resulting similarity score 
# rounded to 4 decimal places? Note: We follow zero-indexing here."""

# #
# # Load the model
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# # Get prompt and Option B from row 0
# prompt = dataset[0]["prompt"]
# option_b = dataset[0]["B"]

# # Generate embeddings
# prompt_embedding = model.encode(prompt, convert_to_tensor=True)
# option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# # Cosine similarity
# similarity = util.cos_sim(prompt_embedding, option_b_embedding)

# print(round(similarity.item(), 4))

In [15]:
# #Question-10:

# """Build two complete ranking pipelines evaluating every row in train.csv.

# Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

# Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for 
# the prompt and all five options. Rank options using cosine similarity to form Top-3 
# predictions.

# First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire 
# training set? 

# Second, count the number of questions for which the correct answer is NOT present in the 
# TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact 
# resulting count?  
# """

# # ---------- MAP@3 ----------
# def apk(actual, predicted, k=3):
#     score = 0.0
#     num_hits = 0.0

#     for i, p in enumerate(predicted[:k]):
#         if p == actual and p not in predicted[:i]:
#             num_hits += 1
#             score += num_hits / (i + 1)

#     return score

# def mapk(actuals, predictions, k=3):
#     return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)


# # ---------- Pipeline 1 : TF-IDF ----------
# tfidf_predictions = []

# for i in range(len(dataset)):
#     prompt = dataset[i]["prompt"]
#     options = [dataset[i][x] for x in ["A", "B", "C", "D", "E"]]

#     texts = [prompt] + options

#     vectorizer = TfidfVectorizer(stop_words="english")
#     X = vectorizer.fit_transform(texts)

#     sims = cosine_similarity(X[0:1], X[1:]).flatten()

#     order = sims.argsort()[::-1][:3]
#     tfidf_predictions.append([["A", "B", "C", "D", "E"][j] for j in order])


# # ---------- Pipeline 2 : MiniLM ----------
# minilm_predictions = []

# for i in range(len(dataset)):
#     prompt = dataset[i]["prompt"]
#     options = [dataset[i][x] for x in ["A", "B", "C", "D", "E"]]

#     prompt_emb = model.encode(prompt, convert_to_tensor=True)
#     option_embs = model.encode(options, convert_to_tensor=True)

#     sims = util.cos_sim(prompt_emb, option_embs).cpu().numpy().flatten()

#     order = sims.argsort()[::-1][:3]
#     minilm_predictions.append([["A", "B", "C", "D", "E"][j] for j in order])


# # ---------- Evaluation ----------
# answers = list(dataset["answer"])

# minilm_map3 = mapk(answers, minilm_predictions)

# count = 0
# for ans, tfidf_top3, minilm_top3 in zip(answers, tfidf_predictions, minilm_predictions):
#     if ans not in tfidf_top3 and ans in minilm_top3:
#         count += 1

# print("MiniLM MAP@3:", round(minilm_map3, 4))
# print("Improved count:", count)

In [16]:
#Question-11:

"""Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to 
facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and 
C as the candidate_labels. What is the probability score given to the top-ranked option? 
(Round to 4 decimal places)."""

# Initialize the zero-shot classifier
classifier = pipeline("zero-shot-classification")

# Prompt from row index 1
text = dataset[1]["prompt"]

# Candidate labels = Options A, B, C
candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]

result = classifier(text, candidate_labels)

print(result["labels"])
print(result["scores"])
print(round(result["scores"][0], 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision c626438 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement an

In [17]:
#Question-12:

"""Run the exact same zero-shot classification as the previous question, but this time pass 
the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous 
question (which uses Softmax) and the sum of the 3 probabilities in this question 
(which uses independent Sigmoids)?"""

result_multi = classifier(
    text,
    candidate_labels,
    multi_label=True
)

print(result_multi["scores"])

softmax_sum = sum(result["scores"])
sigmoid_sum = sum(result_multi["scores"])

print("Softmax sum:", softmax_sum)
print("Sigmoid sum:", sigmoid_sum)
print("Absolute difference:", round(abs(softmax_sum - sigmoid_sum), 4))

[0.00046927202492952347, 2.0635883629438467e-05, 1.9700619304785505e-05]
Softmax sum: 1.0
Sigmoid sum: 0.0005096085278637474
Absolute difference: 0.9995


In [18]:
#Question-13:

"""Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline
("text2text-generation"). Construct the following exact string for row index 0: 
"Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A 
or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output 
returned by the model? """

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

prompt_text = dataset[0]["prompt"]
option_a = dataset[0]["A"]
option_b = dataset[0]["B"]

input_text = (
    f"Question: {prompt_text}. "
    f"Is the correct answer A: {option_a} or B: {option_b}? "
    f"Answer with just the letter A or B."
)

output = generator(input_text, max_new_tokens=5)

print(output)
print(output[0]["generated_text"])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

[{'generated_text': 'B'}]
B


In [19]:
print(transformers.__version__)

4.41.2
